# Filter Spots by ROI

Filters a TrackMate spots CSV, keeping only the spots that fall inside a hand-drawn ROI
mask -- a binary TIFF the same pixel size as the movie (nonzero = inside the ROI).

Used as a preprocessing step before the B cell/macrophage clustering pipeline
(`BCellClustering.ipynb` / `MacrophageClustering.ipynb`), to restrict a movie's spots to a
region of interest (e.g. a single micropattern/well) before downstream analysis.

## Configuration
Edit `SPOTS_PATH`, `MASK_PATH`, and `SUFFIX` below, then run the notebook top to bottom.
Both paths are relative to this notebook's own folder so it stays portable across machines.

In [1]:
# ─── IMPORTS ───────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

# ─── CONFIG ────────────────────────────────────────────────────────────────
BASE_DIR = Path.cwd()

# Example: one replicate's spots CSV + its ROI mask from `Phagy Count Sorted Data`.
# Point these at whichever spots CSV / mask TIFF pair you want to filter.
SPOTS_PATH = BASE_DIR / "Phagy Count Sorted Data" / "90 µm" / "R1" / "20240911_90um_Y_R1_spots_unfiltered.csv"
MASK_PATH = BASE_DIR / "Phagy Count Sorted Data" / "90 µm" / "R1" / "BCellM0_90um_R1_mask_whole.tif"
SUFFIX = "_filtered"

# Filtered output is written here, NOT alongside the source files -- keeps this
# notebook's runs from writing into `Phagy Count Sorted Data`.
OUTPUT_DIR = BASE_DIR / "Testing" / "FilterByROI_output"


## Filtering function

In [2]:
# ─── FILTER SPOTS BY ROI ─────────────────────────────────────────────────────
def points_inside_mask(x, y, mask):
    """Boolean array: True where (x, y), rounded to the nearest pixel, falls
    inside the bounds of `mask` and that pixel is truthy."""
    col_idx = np.round(x).astype(int)
    row_idx = np.round(y).astype(int)
    in_bounds = (row_idx >= 0) & (row_idx < mask.shape[0]) & (col_idx >= 0) & (col_idx < mask.shape[1])
    inside = np.zeros(len(x), dtype=bool)
    inside[in_bounds] = mask[row_idx[in_bounds], col_idx[in_bounds]]
    return inside

def filter_spots_by_roi(spots_path: Path, mask_path: Path, output_dir: Path, suffix: str = "_filtered") -> Path:
    """Keep only the spots in `spots_path` that fall inside `mask_path`'s ROI.
    Saves into `output_dir` (created if needed) with `suffix` appended before the
    extension; returns the output path."""
    if not spots_path.is_file():
        raise FileNotFoundError(f"Spots CSV file not found: {spots_path}")
    if not mask_path.is_file():
        raise FileNotFoundError(f"Mask TIFF file not found: {mask_path}")

    mask = tifffile.imread(mask_path).astype(bool)
    print(f"Loaded ROI mask from {mask_path.name} (shape {mask.shape})")

    spots = pd.read_csv(spots_path, low_memory=False)
    for col in ["POSITION_X", "POSITION_Y"]:
        if col not in spots.columns:
            raise KeyError(f"Column '{col}' not found in {spots_path.name}")
        spots[col] = pd.to_numeric(spots[col], errors="coerce")
    spots = spots.dropna(subset=["POSITION_X", "POSITION_Y"])

    keep = points_inside_mask(spots["POSITION_X"].to_numpy(), spots["POSITION_Y"].to_numpy(), mask)
    filtered_spots = spots.loc[keep]
    print(f"Retained {len(filtered_spots)} spots inside ROI out of {len(spots)} total.")

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{spots_path.stem}{suffix}{spots_path.suffix}"
    filtered_spots.to_csv(out_path, index=False)
    print(f"Filtered spots saved to: {out_path}")
    return out_path


## Run

In [3]:
filter_spots_by_roi(SPOTS_PATH, MASK_PATH, OUTPUT_DIR, SUFFIX)


Loaded ROI mask from BCellM0_90um_R1_mask_whole.tif (shape (1200, 1600))


Retained 88694 spots inside ROI out of 196870 total.


Filtered spots saved to: /Users/brianmah/Claude/MCP Clustering Workspace/Testing/FilterByROI_output/20240911_90um_Y_R1_spots_unfiltered_filtered.csv


PosixPath('/Users/brianmah/Claude/MCP Clustering Workspace/Testing/FilterByROI_output/20240911_90um_Y_R1_spots_unfiltered_filtered.csv')